# Chatbot de reservas basado en reglas — Nivel 0 de la progresión de agentes

**Nivel 0 (sin IA):** sistema de reservas de un restaurante controlado por un menú de
opciones y reglas de negocio explícitas (aforo máximo, validación de fecha/hora). No usa
NLP, similitud semántica ni ningún modelo de lenguaje.

Este caso es intencionadamente el punto de partida de la categoría **Agentes IA**: sirve
de línea base para medir, en los siguientes casos, cuánto aporta introducir similitud
semántica y después un LLM real a la hora de interpretar la intención del usuario.

In [1]:
import pandas as pd

## 1. Configuración inicial — aforo y registro de reservas

In [2]:
# Aforo máximo del restaurante
AFORO_MAXIMO = 50

# DataFrame en memoria con las reservas activas
reservas = pd.DataFrame(
    columns=[
        "Nombre",
        "Fecha",
        "Hora",
        "Personas"
    ]
)

## 2. Motor de reglas — funciones de negocio

In [3]:
def personas_reservadas(fecha, hora):
    if reservas.empty:
        return 0

    filtro = (
        (reservas["Fecha"] == fecha)
        & (reservas["Hora"] == hora)
    )

    return reservas.loc[filtro, "Personas"].sum()


def hacer_reserva():
    global reservas

    nombre = input("Ingrese su nombre: ")
    fecha = input("Fecha (AAAA-MM-DD): ")
    hora = input("Hora (HH:MM): ")
    personas = int(input("Número de personas: "))

    ocupadas = personas_reservadas(fecha, hora)

    if ocupadas + personas <= AFORO_MAXIMO:

        nueva = pd.DataFrame({
            "Nombre": [nombre],
            "Fecha": [fecha],
            "Hora": [hora],
            "Personas": [personas]
        })

        reservas = pd.concat(
            [reservas, nueva],
            ignore_index=True
        )

        print("Reserva realizada con éxito.")

    else:
        print("No hay capacidad disponible.")


def consultar_disponibilidad():
    fecha = input("Fecha (AAAA-MM-DD): ")
    hora = input("Hora (HH:MM): ")

    ocupadas = personas_reservadas(fecha, hora)
    libres = AFORO_MAXIMO - ocupadas

    print(
        f"Disponibilidad para {fecha} a las {hora}: "
        f"{libres} personas."
    )


def cancelar_reserva():
    global reservas

    nombre = input("Ingrese su nombre: ")

    if nombre in reservas["Nombre"].values:

        reservas = reservas[
            reservas["Nombre"] != nombre
        ]

        print("Reserva cancelada.")

    else:
        print("No se encontró ninguna reserva con ese nombre.")


def mostrar_reservas():
    if reservas.empty:
        print("No hay reservas.")

    else:
        print("\nReservas actuales:")
        print(reservas)

## 3. Bucle conversacional basado en menú de opciones

El "chatbot" no interpreta lenguaje natural: el usuario elige una opción numérica y el
programa ejecuta la función de reglas correspondiente.

In [4]:
def chatbot():

    print("=" * 50)
    print("Bienvenido al sistema de reservas del restaurante.")
    print("=" * 50)

    while True:

        print("\nOpciones:")
        print("1. Hacer una reserva")
        print("2. Consultar disponibilidad")
        print("3. Cancelar una reserva")
        print("4. Mostrar todas las reservas")
        print("5. Salir")

        opcion = input("Seleccione una opción (1-5): ")

        if opcion == "1":
            hacer_reserva()

        elif opcion == "2":
            consultar_disponibilidad()

        elif opcion == "3":
            cancelar_reserva()

        elif opcion == "4":
            mostrar_reservas()

        elif opcion == "5":
            print(
                "Gracias por usar el sistema de reservas. "
                "¡Hasta luego!"
            )
            break

        else:
            print(
                "Opción no válida. "
                "Por favor, intente de nuevo."
            )


# Descomentar para ejecutar en modo interactivo (requiere terminal con input()):
# chatbot()